# Multiple Linear Regression: economics example

**Goal:** use interest rate and unemployment rate to estimate an index price.

Simple regression uses one clue. Multiple regression uses two or more clues to predict one number. The small classroom data is included here. It is not a current forecast or proof of cause and effect.

## 1. Bring in the tools

Pandas stores the table, Matplotlib draws graphs, and scikit-learn learns and checks the model.

In [ ]:
import pandas as pd  # Organize examples in a table.
import matplotlib.pyplot as plt  # Draw the graphs.

from sklearn.model_selection import train_test_split  # Save rows for testing.
from sklearn.linear_model import LinearRegression  # Learn the prediction rule.
from sklearn.metrics import mean_absolute_error, r2_score  # Measure test mistakes.

## 2. Make the classroom data table

Each row is one recorded situation. The two rate columns are clues; index_price is the answer. These few rows are for learning, not a dependable forecast.

In [ ]:
data = {  # Values in the same position belong to one situation.
    "interest_rate": [2.75, 2.5, 2.5, 2.5, 2.5, 2.5, 2.5, 2.25,
                      2.25, 2.25, 2.0, 2.0, 2.0, 1.75, 1.75, 1.75,
                      1.75, 1.75, 1.75, 1.75, 1.75, 1.75, 1.75, 1.75],
    "unemployment_rate": [5.3, 5.3, 5.3, 5.3, 5.4, 5.6, 5.5, 5.5,
                          5.5, 5.6, 5.7, 5.9, 6.0, 5.9, 5.8, 6.1,
                          6.2, 6.1, 6.1, 6.1, 5.9, 6.2, 6.2, 6.1],
    "index_price": [1464, 1394, 1357, 1293, 1256, 1254, 1234, 1195,
                    1159, 1167, 1130, 1075, 1047, 965, 943, 958,
                    971, 949, 884, 866, 876, 822, 704, 719]
}
df = pd.DataFrame(data)  # Turn the lists into named columns.
print("Number of situations:", len(df))  # Count the examples.
df.head()  # Preview the first five.

**Read the data code:** each list becomes one column. A row combines one interest rate, one unemployment rate, and its index price. Putting the values here means no separate file is needed.

## 3. Choose clues and answer

X holds both input columns. y holds the price we want to predict. Each row of X has two clues.

In [ ]:
X = df[["interest_rate", "unemployment_rate"]]  # The two input clues.
y = df["index_price"]  # The answer to predict.

print("Input shape (rows, clues):", X.shape)  # Check examples and clues.
print("Number of answers:", len(y))  # One answer belongs to each row.

## 4. Look at each clue

The graphs show how each input moves alongside price. A pattern can help prediction, but moving together does not prove cause and effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # Make two side-by-side graphs.

axes[0].scatter(df["interest_rate"], y, color="teal")  # Rate one vs price.
axes[0].set_xlabel("Interest rate")
axes[0].set_ylabel("Index price")
axes[0].set_title("Interest rate and price")

axes[1].scatter(df["unemployment_rate"], y, color="darkorange")  # Rate two vs price.
axes[1].set_xlabel("Unemployment rate")
axes[1].set_ylabel("Index price")
axes[1].set_title("Unemployment and price")

for ax in axes:  # Add guide lines to both graphs.
    ax.grid(alpha=0.25)
plt.tight_layout()  # Keep labels from overlapping.
plt.show()

## 5. Save some rows for testing

The model learns from training rows. Test rows stay hidden until the end, like a quiz. There are few rows, so the test score can jump around.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)  # Learn from 75%; keep 25% hidden.

print("Rows for learning:", len(X_train))
print("Hidden test rows:", len(X_test))

## 6. Teach the model from training rows

The model combines a starting value and the effects of both clues:

**predicted price = start + interest effect × interest rate + unemployment effect × unemployment rate**

In [ ]:
model = LinearRegression()  # Create an untrained multiple-input model.
model.fit(X_train, y_train)  # Learn using training rows only.
print("The model is trained.")

## 7. Understand the learned numbers

A coefficient describes how the model's guess changes when that input rises by one unit **while the other input stays the same**. It describes a pattern in this dataset, not a cause.

In [ ]:
print(f"Starting value: {model.intercept_:.1f}")  # Model's learned starting point.

for name, weight in zip(X.columns, model.coef_):  # Pair each clue with its weight.
    print(f"{name}: effect per one-unit increase = {weight:.1f}")  # Show its effect.

## 8. Check guesses on hidden rows

MAE is average mistake size in price units; smaller is better. R-squared compares with always guessing the average price; nearer 1 is usually better. There are only six test rows, so do not treat the score as a reliable economic result.

In [ ]:
price_guesses = model.predict(X_test)  # Guess prices for hidden inputs.
mae = mean_absolute_error(y_test, price_guesses)  # Average absolute mistake.
r2 = r2_score(y_test, price_guesses)  # Compare with the average-price guess.

print(f"Average test mistake (MAE): {mae:.1f} price units")
print(f"Test R-squared: {r2:.2f}")

## 9. Compare real prices with guesses

A dot on the dashed line is a perfect guess. Farther away means a larger mistake.

In [ ]:
plt.scatter(y_test, price_guesses, color="crimson", s=70)  # Real value vs guess.

low = min(y_test.min(), price_guesses.min())  # Bottom of the graph.
high = max(y_test.max(), price_guesses.max())  # Top of the graph.
plt.plot([low, high], [low, high], "--", color="steelblue", label="Perfect guess")

plt.xlabel("Real index price")
plt.ylabel("Model guess")
plt.title("Were the hidden-row guesses close?")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 10. Guess one new situation

For a new guess, include **both** clues with the same column names. This code shows prediction only; it is not a real economic forecast.

In [ ]:
new_situation = pd.DataFrame({  # Make one row with both required clues.
    "interest_rate": [2.25],
    "unemployment_rate": [5.6]
})
new_price_guess = model.predict(new_situation)[0]  # Ask the model for its first guess.
print(f"Example predicted index price: {new_price_guess:.1f}")

## Quick revision

- Multiple regression uses several clues to predict one number.
- X is the input table; y is the answer.
- A coefficient describes the model's change when that clue changes and others stay fixed.
- Train on one group; check on hidden rows.
- MAE is average mistake size; R-squared compares with an average-only guess.
- A tiny sample cannot prove cause or make reliable forecasts.

**Remember:** several clues go in; one number comes out.